# Compiled vs pure-Python backend

Penicillium evaluates the model either with Numba (`backend="numba"`, the default when the `fast` extra is
installed) or in plain Python (`backend="python"`). Both use the same SciPy solver, so the ratio of their run times
measures what compiling the model saves.

The pure-Python backend is also a proxy for any IndPenSim port that evaluates the model in Python and integrates it
with SciPy: its run times show what such a port costs per batch. It is an estimate, not a measurement of another
package: solvers and code still differ.

Setup: default 230 h batches, Raman spectra off, one batch at a time on one core, mean of three seeds. The first
Numba run (compilation, cached on disk) is excluded. The pure-Python backend is timed at Penicillium's default
tolerance (`rtol=1e-5`, `atol=1e-7`) and at two common alternatives: `rtol=1e-3` (MATLAB's `ode15s` default, used by
IndPenSim) and `rtol=1e-6`.

In [1]:
import os
import platform
import time

import pandas as pd

import penicillium
from penicillium import RamanMode, SimulationConfig, Simulator

SEEDS = [1, 2, 3]
CONFIG = SimulationConfig(raman_mode=RamanMode.OFF)
print(
    f"Penicillium {penicillium.__version__}, Python {platform.python_version()}, "
    f"{platform.processor() or platform.machine()}, {os.cpu_count()} logical CPUs, {platform.system()}"
)

Penicillium 0.1.0.dev0, Python 3.14.4, Intel64 Family 6 Model 158 Stepping 9, GenuineIntel, 8 logical CPUs, Windows


In [2]:
def seconds_per_batch(backend, rtol=None, atol=None):
    sim = Simulator(config=CONFIG, backend=backend, rtol=rtol, atol=atol)
    times = []
    for seed in SEEDS:
        t = time.perf_counter()
        sim.run(seed=seed)
        times.append(time.perf_counter() - t)
    return times


Simulator(config=CONFIG, backend="numba").run(seed=0)  # warm-up: loads the compiled model

cases = {
    "numba, rtol 1e-5 (default)": ("numba", None, None),
    "python, rtol 1e-5 (default)": ("python", None, None),
    "python, rtol 1e-3": ("python", 1e-3, 1e-6),
    "python, rtol 1e-6": ("python", 1e-6, 1e-9),
}
speed = pd.DataFrame(
    {name: seconds_per_batch(*args) for name, args in cases.items()}, index=[f"seed {s}" for s in SEEDS]
).T
speed["mean [s]"] = speed.mean(axis=1)
speed["vs numba"] = speed["mean [s]"] / speed["mean [s]"].iloc[0]
speed.round(2)

,seed 1,seed 2,seed 3,mean [s],vs numba
"numba, rtol 1e-5 (default)",0.69,0.61,0.86,0.72,1.00
"python, rtol 1e-5 (default)",18.09,12.61,11.03,13.91,19.22
"python, rtol 1e-3",15.95,13.87,15.61,15.15,20.94
"python, rtol 1e-6",22.19,16.36,15.75,18.10,25.02


Compiling the model is what makes Penicillium fast: with the same solver and tolerance, the pure-Python
backend takes many times longer per batch. A port that evaluates the model in Python with a SciPy solver should
expect run times in the range of the pure-Python rows, depending on its tolerance.